# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ansh-Saraswat-PC/FlyRank/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1: "Refreshing stale content leads to a 30% average recovery in organic sessions."
Methodology Question: How was the "refresh" label generated? If the label relies on a manual flag from editors who only bother to tag successful/completed updates (survivorship bias), the 30% recovery claim might be artificially inflated.
Finding 2: "Pages sitting at position 11 have the highest statistical probability of moving to Page 1 within the next 30 days."
Methodology Question: Does the validation design use a strict time-aware split? If the model used a randomized K-fold split across all dates, future ranking data (e.g., knowing a page jumped to position 8 on March 25th) might have leaked into the training set (March 15th), invalidating the predictive claim.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
pass

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

My Week 5 model used a naive random split. The danger is that the model could memorize the specific performance patterns of massive domains (clients) in the training set and fail when applied to a completely new website.
To fix this, I am implementing a Grouped Split by client_hash_id. This guarantees that if a client's URLs are in the test set, zero URLs from that client exist in the training set, forcing the model to prove it actually learned universal SEO principles, not just domain-specific quirks.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
import pandas as pd
import numpy as np
import os
import getpass
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

# 1. Setup & Auth
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
hf_token = os.environ.get('HF_TOKEN')
if not hf_token:
    hf_token = getpass.getpass("HF_TOKEN not found in env. Paste your Hugging Face READ token here: ")
con.execute(f"CREATE SECRET (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

# 2. Load Aggregated Data (NOW WITH client_hash_id)
hf_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
df_model = con.sql(f"""
    SELECT 
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) as impressions,
        SUM(gsc_clicks) as clicks,
        AVG(gsc_avg_position) as position,
        SUM(sessions_organic) as sessions
    FROM '{hf_path}'
    WHERE gsc_impressions > 100
    GROUP BY client_hash_id, content_hash_id
""").df()

# 3. Define the Proxy Target
df_model['ctr'] = df_model['clicks'] / df_model['impressions']
imp_median = df_model['impressions'].median()
df_model['target_opportunity'] = ((df_model['impressions'] > imp_median) & (df_model['ctr'] < 0.01)).astype(int)

# 4. Features & The "Before" Split (Random)
features = ['impressions', 'clicks', 'position', 'sessions']
X = df_model[features]
y = df_model['target_opportunity']

X_train_rand, X_test_rand, y_train_rand, y_test_rand, idx_train_rand, idx_test_rand = train_test_split(
    X, y, df_model.index, test_size=0.2, random_state=42
)
rf_rand = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_rand.fit(X_train_rand, y_train_rand)
df_test_rand = df_model.loc[idx_test_rand].copy()
df_test_rand['ml_score_rand'] = rf_rand.predict_proba(X_test_rand)[:, 1]
precision_rand = df_test_rand.sort_values('ml_score_rand', ascending=False).head(100)['target_opportunity'].mean()

# 5. The "After" Split (Grouped by Client)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df_model['client_hash_id']))

X_train_grp, X_test_grp = X.iloc[train_idx], X.iloc[test_idx]
y_train_grp, y_test_grp = y.iloc[train_idx], y.iloc[test_idx]

rf_grp = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_grp.fit(X_train_grp, y_train_grp)

df_test_grp = df_model.iloc[test_idx].copy()
df_test_grp['ml_score_grp'] = rf_grp.predict_proba(X_test_grp)[:, 1]
precision_grp = df_test_grp.sort_values('ml_score_grp', ascending=False).head(100)['target_opportunity'].mean()

print("--- VALIDATION AUDIT: Precision@100 ---")
print(f"Before: Random Split (Naive):     {precision_rand:.2f}")
print(f"After:  Client-Grouped Split:     {precision_grp:.2f}")

HF_TOKEN not found in env. Paste your Hugging Face READ token here:  ········


--- VALIDATION AUDIT: Precision@100 ---
Before: Random Split (Naive):     1.00
After:  Client-Grouped Split:     1.00


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

I audited the final feature set (impressions, clicks, position, sessions) against the proxy target. Because all features are strictly aggregated within the March 2026 window, and the target is mathematically derived from that exact same window (no future look-aheads), time leakage is structurally impossible. A correlation check confirms no single feature acts as a perfectly leaked proxy for the target.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("--- LEAKAGE AUDIT ---")
print("Target Definition: (impressions > median) AND (ctr < 0.01)\n")

print("Feature Importances (Grouped Model):")
for name, importance in zip(features, rf_grp.feature_importances_):
    print(f"{name}: {importance:.4f}")

# Check for perfect correlation (1.00) which implies a leaked label
print("\nCorrelation with Target:")
corr_matrix = df_model[features + ['target_opportunity']].corr()['target_opportunity'].drop('target_opportunity')
print(corr_matrix)

--- LEAKAGE AUDIT ---
Target Definition: (impressions > median) AND (ctr < 0.01)

Feature Importances (Grouped Model):
impressions: 0.8037
clicks: 0.1477
position: 0.0104
sessions: 0.0381

Correlation with Target:
impressions    0.437004
clicks         0.183697
position       0.063873
sessions       0.157484
Name: target_opportunity, dtype: float64


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Original Bold Claim: "Our Random Forest model accurately predicts which pages will lose traffic, guaranteeing a higher ROI for content refresh campaigns."
Safe Rewrite: "Our Random Forest model is observed to identify pages with a severe impression-to-click disparity. This provides a directional, decision-support signal to help editorial teams prioritize their content review workflows."

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
pass

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.